In [0]:
storage_account = "insurancestorage01"

spark.conf.set(
"fs.azure.account.key."+storage_account+".dfs.core.windows.net",
"azure_Storage_key"
)

spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

silver_adls_path = "abfss://silver@"+storage_account+".dfs.core.windows.net/"
gold_adls_path = "abfss://gold@"+storage_account+".dfs.core.windows.net/"

In [0]:
from pyspark.sql.functions import col

customers = spark.read.format("delta").load(silver_adls_path+"/customer_master")

dim_customer = (
    customers
    .select(
        "customer_id",
        "gender",
        "age",
        "city",
        "state",
        "income_band"
    )
    .dropDuplicates(["customer_id"])
)

dim_customer.write.format("delta").mode("overwrite").save(gold_adls_path+"/dim_customer")